# Ingestion 4: Load into DuckDB with dlt

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/ingestion/ingestion-04-dlt.ipynb)

Run a real dlt pipeline with a local DuckDB destination.

In [ ]:
# Setup: install notebook dependencies in Colab.
!pip install -q dlt[duckdb] duckdb~=2.0

## Setup and sample data

Two input orders have integer keys and amounts in cents.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

import dlt
import duckdb

workspace = TemporaryDirectory()
base = Path(workspace.name)
database = base / "landing.duckdb"
orders = [{"id": 1, "cents": 1200}, {"id": 2, "cents": 800}]

## Run a resource and inspect the destination

A resource yields records. A dlt pipeline extracts them, normalizes values and writes tables to its destination.

**Where it matters**

Use the load metadata and destination tables to troubleshoot a batch.

**Watch out for**

Append loads can duplicate input if delivery is replayed. Choose the write disposition and keys deliberately.

In [ ]:
pipeline = dlt.pipeline(
    pipeline_name="orders_demo",
    destination=dlt.destinations.duckdb(credentials=str(database)),
    dataset_name="raw",
    pipelines_dir=str(base / "pipelines"),
)
pipeline.run(orders, table_name="orders", write_disposition="replace")
with duckdb.connect(str(database)) as connection:
    print(connection.execute("select id, cents from raw.orders order by id").fetchall())

**What to notice:** The destination contains two rows with IDs 1 and 2. This bounded replacement assumes the list is the whole table.

## Check yourself

### Exercise 1: Apply the pattern

**Difficulty:** Easy · about 5 min

If the next input only updates ID 2, should you replace the whole table with it?

In [ ]:
# Try your solution here.

In [ ]:
print("No. Use a keyed merge for a partial update.")

## Key takeaways

- Use the load metadata and destination tables to troubleshoot a batch.

**Next:** Continue the learning path with [PySpark: Start here](../pyspark/pyspark-00-start-here.ipynb). To validate what you load, see [Validate data with Pandera](../concepts/pandera-data-quality.ipynb).